In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

INPUT_FILE = "TextToGloss_normalized.xlsx"
TEXT_COL = "text_ar"
GLOSS_COL = "gloss_ar"

df = pd.read_excel(INPUT_FILE)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(10)

Shape: (128, 3)
Columns: ['text_ar', 'gloss_ar', 'id']


,text_ar,gloss_ar,id
0,انتبه,انتبه خطر تمام,1
1,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,2
2,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,3
3,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,4
4,معلقة كبيرة 3 مرة .,3 مرة ملعقة - كبيرة شراب تمام,5
5,6 شهر,6 شهر,6
6,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,7
7,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,8
8,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,9
9,بعد الاكل,اكل خلص فورا دواء تمام,10


In [2]:
required_cols = [TEXT_COL, GLOSS_COL]
missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("All required columns are present.")

All required columns are present.


In [3]:
df = df.dropna(subset=[TEXT_COL, GLOSS_COL]).copy()

df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[GLOSS_COL] = df[GLOSS_COL].astype(str).str.strip()

df = df[(df[TEXT_COL] != "") & (df[GLOSS_COL] != "")].copy()

# حذف التكرارات احتياطياً
df = df.drop_duplicates(subset=[TEXT_COL, GLOSS_COL]).reset_index(drop=True)

print("Shape after final cleanup:", df.shape)
df.head(10)

Shape after final cleanup: (128, 3)


,text_ar,gloss_ar,id
0,انتبه,انتبه خطر تمام,1
1,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,2
2,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,3
3,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,4
4,معلقة كبيرة 3 مرة .,3 مرة ملعقة - كبيرة شراب تمام,5
5,6 شهر,6 شهر,6
6,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,7
7,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,8
8,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,9
9,بعد الاكل,اكل خلص فورا دواء تمام,10


In [5]:
df = df.reset_index(drop=True)

df["source_type"] = "original"
df["aug_method"] = "none"
df["parent_id"] = df["id"]
df["is_augmented"] = 0

print("Shape after adding metadata columns:", df.shape)
df.head(10)

Shape after adding metadata columns: (128, 7)


,text_ar,gloss_ar,id,source_type,aug_method,parent_id,is_augmented
0,انتبه,انتبه خطر تمام,1,original,none,1,0
1,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,2,original,none,2,0
2,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,3,original,none,3,0
3,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,4,original,none,4,0
4,معلقة كبيرة 3 مرة .,3 مرة ملعقة - كبيرة شراب تمام,5,original,none,5,0
5,6 شهر,6 شهر,6,original,none,6,0
6,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,7,original,none,7,0
7,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,8,original,none,8,0
8,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,9,original,none,9,0
9,بعد الاكل,اكل خلص فورا دواء تمام,10,original,none,10,0


In [6]:
RANDOM_STATE = 42

TRAIN_SIZE = 0.80
VALID_SIZE = 0.10
TEST_SIZE  = 0.10

assert abs((TRAIN_SIZE + VALID_SIZE + TEST_SIZE) - 1.0) < 1e-9, "Split ratios must sum to 1.0"

print("Train:", TRAIN_SIZE)
print("Valid:", VALID_SIZE)
print("Test :", TEST_SIZE)

Train: 0.8
Valid: 0.1
Test : 0.1


In [7]:
train_df, temp_df = train_test_split(
    df,
    test_size=(1 - TRAIN_SIZE),
    random_state=RANDOM_STATE,
    shuffle=True
)

print("Train shape:", train_df.shape)
print("Temp shape :", temp_df.shape)

Train shape: (102, 7)
Temp shape : (26, 7)


In [8]:
# temp = 20% من البيانات
# نريد تقسيمها إلى 10% valid و10% test
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("Train shape:", train_df.shape)
print("Valid shape:", valid_df.shape)
print("Test shape :", test_df.shape)

Train shape: (102, 7)
Valid shape: (13, 7)
Test shape : (13, 7)


In [9]:
final_cols = [
    "id",
    TEXT_COL,
    GLOSS_COL,
    "source_type",
    "aug_method",
    "parent_id",
    "is_augmented"
]

# لو عندك أعمدة إضافية، نضيفها بالنهاية
extra_cols = [c for c in df.columns if c not in final_cols]
final_cols_all = final_cols + extra_cols

train_df = train_df[final_cols_all].sort_values("id").reset_index(drop=True)
valid_df = valid_df[final_cols_all].sort_values("id").reset_index(drop=True)
test_df  = test_df[final_cols_all].sort_values("id").reset_index(drop=True)

print("Final train columns:", train_df.columns.tolist())

Final train columns: ['id', 'text_ar', 'gloss_ar', 'source_type', 'aug_method', 'parent_id', 'is_augmented']


In [10]:
print("Total samples :", len(df))
print("Train samples :", len(train_df))
print("Valid samples :", len(valid_df))
print("Test samples  :", len(test_df))

print("\nRatios:")
print("Train ratio:", round(len(train_df) / len(df), 4))
print("Valid ratio:", round(len(valid_df) / len(df), 4))
print("Test ratio :", round(len(test_df) / len(df), 4))

Total samples : 128
Train samples : 102
Valid samples : 13
Test samples  : 13

Ratios:
Train ratio: 0.7969
Valid ratio: 0.1016
Test ratio : 0.1016


In [11]:
train_df.to_excel("train.xlsx", index=False)
valid_df.to_excel("valid.xlsx", index=False)
test_df.to_excel("test.xlsx", index=False)

print("Saved:")
print("- train.xlsx")
print("- valid.xlsx")
print("- test.xlsx")

Saved:
- train.xlsx
- valid.xlsx
- test.xlsx


In [12]:
with pd.ExcelWriter("split_dataset.xlsx", engine="openpyxl") as writer:
    train_df.to_excel(writer, sheet_name="train", index=False)
    valid_df.to_excel(writer, sheet_name="valid", index=False)
    test_df.to_excel(writer, sheet_name="test", index=False)

print("Saved: split_dataset.xlsx")

Saved: split_dataset.xlsx


In [13]:
print("TRAIN SAMPLE")
display(train_df.head(10))

print("VALID SAMPLE")
display(valid_df.head(10))

print("TEST SAMPLE")
display(test_df.head(10))

TRAIN SAMPLE


,id,text_ar,gloss_ar,source_type,aug_method,parent_id,is_augmented
0,1,انتبه,انتبه خطر تمام,original,none,1,0
1,2,حبة 3 مرة قبل الاكل بنص ساعة .,3 مرة اكل قبل نص ساعة حبة,original,none,2,0
2,3,حبة 3 مرة قبل الاكل .,3 مرة اكل قبل حبة,original,none,3,0
3,4,حبة 3 مرة قبل الاكل بساعة .,3 مرة اكل قبل ساعة حبة,original,none,4,0
4,6,6 شهر,6 شهر,original,none,6,0
5,7,نقطتين بكل طرف,اذن يمين نقطة - نقطة اذن يسار نقطة - نقطة,original,none,7,0
6,8,لمدة ثلاثة يوم,استمرار ثلاثة يوم بس تمام,original,none,8,0
7,9,خده ورا الاكل,اكل بعد حبة تمام معدة فاضية لا,original,none,9,0
8,10,بعد الاكل,اكل خلص فورا دواء تمام,original,none,10,0
9,13,"ممكن يعملك شوية اسهال , لا تقلق .",انتبه تمام جسم تعب تمام اسهال امساك دوخة تمام,original,none,13,0


VALID SAMPLE


,id,text_ar,gloss_ar,source_type,aug_method,parent_id,is_augmented
0,11,مع اكلة دسمة .,اكل دسم ثقيل لازم تمام بيض حليب زيت تمام دواء ...,original,none,11,0
1,19,"حبة قبل الاكل بنص ساعة ,",ثلاثة مرة حبة قبل اكل انتظر نص ساعة,original,none,19,0
2,32,بخاخ مرتين باليوم,دواء انف بخ تمام يمين واحد يسار واحد تمام تكرا...,original,none,32,0
3,46,لونه وريحته بس يتغيروا كبه بكون انتزع .,دواء لون غير تمام ريحة سيء تمام دواء انتزع تما...,original,none,46,0
4,70,"حبة بالنهار بعد الاكل لمدة ثلاثة يوم ,",كل يوم حبة اكل بعد تمام استمرار ثلاثة يوم بس,original,none,70,0
5,81,حبة 5 مرة باليوم,يوم حبة حبة كم سوال خمسة,original,none,81,0
6,85,"حبة تحت اللسان : لا تبلعها , اتركها تذوب لحالها",حبة تحت لسان بلع لا ممنوع تمام ذوبان وحدها لازم,original,none,85,0
7,95,فيك تخلط الدواء مع شوية عصير,دواء مع عصير خلط عادي,original,none,95,0
8,97,غير مكان الحقن كل يوم,ابرة مكان تغيير يوميا نفس مكان لا,original,none,97,0
9,99,استمر بالتنفس بعمق حتي ينتهي المحلول تماما .,نفس عميق استمرار دواء كلو خلص تمام,original,none,99,0


TEST SAMPLE


,id,text_ar,gloss_ar,source_type,aug_method,parent_id,is_augmented
0,5,معلقة كبيرة 3 مرة .,3 مرة ملعقة - كبيرة شراب تمام,original,none,5,0
1,12,قبل او بعد الاكل,اكل قبل اكل بعد تمام عادي دواء خذ اي وقت,original,none,12,0
2,20,عند الضرورة .,جسم تعب ضرورة بس دواء,original,none,20,0
3,27,لازم تقطعيه,دواء تمام وقف - فورا,original,none,27,0
4,28,استمر ع البخاخ اسبوعين تلاتة,دواء استمرار ثلاثة اسبوع كامل تمام دواء هدف را...,original,none,28,0
5,37,بلش فيه فورا,دواء بداية الان تمام,original,none,37,0
6,41,"دير بالك من الضوء , خليه بكرتونته دايما .",دواء داخل كرتونة تمام ضوء شمس لا - خطر,original,none,41,0
7,56,ويفضل تبعد عن الشغلة الباردة,شرب بارد لا - ممنوع,original,none,56,0
8,57,حصرا الصبح,صباح بكير حبة تمام لازم,original,none,57,0
9,65,القطرة امنة,قطرة امن مشاكل - جسم - لا,original,none,65,0


In [14]:
train_ids = set(train_df["id"])
valid_ids = set(valid_df["id"])
test_ids  = set(test_df["id"])

print("Train ∩ Valid:", len(train_ids & valid_ids))
print("Train ∩ Test :", len(train_ids & test_ids))
print("Valid ∩ Test :", len(valid_ids & test_ids))

Train ∩ Valid: 0
Train ∩ Test : 0
Valid ∩ Test : 0
